# 04 · 流式系统:从分类器到唤醒词产品(Chunk 3 下半场)

test F1 0.981 的分类器放进连续音频流,第一次测量得到 **FA/hour = 336**。
本篇记录把这个数字降到 **2.0** 的三轮迭代——每一轮都是"假设 → 可证伪验证 → 修复 → 测量"。

## 系统架构(`src/my_kws/streaming.py`)

- **环形缓冲**:1.0 s 窗口(匹配训练输入)、0.1 s hop → 10 次推理/秒
- **双重发火抑制**:`min_consecutive`(连续 N 个 hop 过阈值才触发,滤单 hop 尖峰)
  + `refractory_s`(触发后不应期,一句话只报一次)
- **可测试性**:`score_fn` 为注入依赖,7 个单元测试用脚本化分数验证全部触发逻辑,
  不加载模型、毫秒级完成(`tests/test_streaming.py`)
- **评测方法**:合成连续流——`_background_noise_` 随机增益拼接 30 分钟,
  60 条 **test split** 的 "yes" 按指定 SNR **混叠**(非拼接)进已知时刻;
  报 recall + FA/hour 两个产品级指标
- 特征为每窗全量重算(相邻窗 90% 帧重叠,增量 STFT 留作后续优化);
  即便如此整体 ~75× 实时,单核推理 0.15 ms 占 1 s 窗口预算的 0.015%

## 三轮迭代:钟摆叙事

| 版本 | 训练数据变更 | 流式 recall | FA/hour | 诊断 |
|---|---|---|---|---|
| v1 | 干净正例 + 单词负例 | 0.850 | **336** | 训练负例全是"说出的单词",模型从未见过"无人说话"——纯噪声是分布外输入,狂发火 |
| v2 | + 4,000 条纯噪声负例 | **0.417** | 0 | 矫枉过正:缓存管线早已丢掉波形加噪,正例全是干净 yes;模型学会"噪声特征→0"后连带压制了带噪正例(nb02 的 caveat 应验) |
| v3 | + 6,456 条带噪正例(仅 train split,SNR 0–20 dB)| 0.933 | 40 | 回正;残余 FA 源自 SNR≈0 的"正例"太像噪声,把边界又拉回噪声领地 |

两次数据修复用的是同一个零改动技巧:.npy 直接写进特征缓存目录——
目录名决定 label(`_noise_`→0,`yes`→1),文件名不在官方 split 列表→自动进 train。
带噪正例脚本含**泄漏防护**:只增强 train-split 文件,val/test 的 yes 绝不进训练。

## 第四轮:阈值旋钮(免费的)

v3 的 hit 分数贴近 1.0 而 FA 分数徘徊在 0.86–0.95 → 阈值上移应能低成本切 FA。
同 seed 同一条流,严格可比:

| 操作点 | recall | FA/hour |
|---|---|---|
| 0.855 / 2 hop | 0.933 | 40 |
| 0.95 / 2 hop | 0.933 | 10 ← recall 零损失砍掉 3/4 的 FA |
| 0.98 / 2 hop | 0.900 | 4 |
| **0.98 / 3 hop(定稿)** | **0.900** | **2.0** |

0.855→0.95 区间 recall 纹丝不动,证实 hit 与 FA 的分数带几乎不重叠。

## 最终验收(live demo,麦克风实测)

- "yes" ×6:**全部命中**,分数均 ≥0.999
- "yeah"/"yea":**不再触发**(v1 时代会误触)——带噪正例训练 + 0.98 阈值的联合效果;
  demo 中观察到 0.874 的孤立峰,恰落在新旧阈值之间,被 0.98 挡下
- 片段指标与系统指标自洽:test R 0.895 @0.98 ↔ 流式 recall 0.900

In [1]:
# 可重跑:流式基准(同 seed 可比;完整 30 min 约 25 s)
# uv 环境下也可直接命令行: uv run python -m my_kws.streaming --minutes 30
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'my_kws.streaming', '--minutes', '10',
     '--threshold', '0.98', '--min-consecutive', '3'],
    capture_output=True, text=True, cwd='..' if not __import__('pathlib').Path('models').exists() else '.')
print(r.stdout or r.stderr)

[stream] building 10 min stream, 60 injected keywords @ SNR 10.0 dB ...

[result] stream 10.0 min | processed at 74.5x realtime
  injected keywords : 60
  hits              : 58  (recall 0.967)
  false accepts     : 1  (FA/hour = 6.00)
  total detections  : 59



In [2]:
# 可重跑:操作曲线扫描(4 个操作点 × 10 min 流)
import subprocess, sys
from pathlib import Path
cwd = '..' if not Path('models').exists() else '.'
print(f"{'threshold':>10s} {'consec':>7s} {'recall':>7s} {'FA/hour':>8s}")
for th, mc in [(0.855, 2), (0.95, 2), (0.98, 2), (0.98, 3)]:
    r = subprocess.run(
        [sys.executable, '-m', 'my_kws.streaming', '--minutes', '10',
         '--threshold', str(th), '--min-consecutive', str(mc)],
        capture_output=True, text=True, cwd=cwd)
    rec = fa = '?'
    for line in r.stdout.splitlines():
        if 'recall' in line: rec = line.split('recall')[1].strip(' ()')
        if 'FA/hour' in line: fa = line.split('FA/hour =')[1].strip(' ()')
    print(f"{th:>10.3f} {mc:>7d} {rec:>7s} {fa:>8s}")

 threshold  consec  recall  FA/hour
     0.855       2   0.983    48.00
     0.950       2   0.983    24.00
     0.980       2   0.967    12.00
     0.980       3   0.967     6.00


In [4]:
# 可重跑:部署配置审计——模型 metadata 必须与代码默认值一致
import onnx
from pathlib import Path
if not Path('models').exists():
    import os; os.chdir('..')
m = onnx.load('models/dscnn_int8.onnx')
meta = {p.key: p.value for p in m.metadata_props}
print('ONNX metadata:', meta)

from my_kws.streaming import StreamingDetector
det_defaults = StreamingDetector.__dataclass_fields__
code_th = det_defaults['threshold'].default
code_mc = det_defaults['min_consecutive'].default
print(f'code defaults: threshold={code_th}, min_consecutive={code_mc}')
assert str(code_th) == meta.get('deployment_threshold'), '阈值漂移!metadata 与代码不一致'
assert str(code_mc) == meta.get('min_consecutive_hops'), 'min_consecutive 漂移!'
print('配置一致性 PASSED')

ONNX metadata: {'onnx.infer': 'onnxruntime.quant', 'onnx.quant.pre_process': 'onnxruntime.quant', 'deployment_threshold': '0.98', 'min_consecutive_hops': '3', 'window_s': '1.0', 'hop_s': '0.1'}
code defaults: threshold=0.98, min_consecutive=3
配置一致性 PASSED


---

## 本篇问题与修复编年史

| # | 问题 | 根因 | 修复 | 教训 |
|---|------|------|------|------|
| 1 | **test F1 0.981 的模型在连续流上 FA/hour = 336** | 训练负例全为切好的单词,"无人说话"从未作为类别出现;片段评测的世界里每个样本都是词,真实世界 99% 的时间没人说话 | 4,000 条纯噪声负例注入缓存 | **片段指标 ≠ 系统指标;FA/hour 必须在连续流上测,而连续流会暴露训练分布里根本不存在的输入类型** |
| 2 | 修复 #1 后 recall 0.850 → 0.417 | 缓存管线无波形加噪 → 正例全干净;模型学到"噪声→0"后连带压制带噪正例(nb02 记录的 caveat 应验) | 6,456 条带噪正例(SNR 0–20 dB,仅 train split) | **修一个分布空洞可能撕开另一个;每次数据修复后全指标重测,而不是只看被修的那个** |
| 3 | 带噪正例脚本的泄漏风险 | 若增强 val/test 的 yes 文件并入训练,即数据泄漏 | 脚本内置 held-out 列表排除 | 任何"生成训练数据"的脚本都要显式回答:源文件属于哪个 split? |
| 4 | v3 FA 回弹到 40 | SNR≈0 的"带噪正例"几乎就是噪声,教坏了边界;另 pos_weight 名义 25.3 实际应 ~9(缓存正例已 3 倍,compute_pos_weight 只数原始 wav),过度偏向正类 | 阈值 0.855→0.98 + min_consecutive 2→3,免费降到 2.0;数据层修复(--snr-low 5、pos_weight 数缓存)留作后续 | **先拧免费旋钮(阈值/后处理),再动昂贵旋钮(数据/重训);hit 与 FA 分数带不重叠时,阈值就是全部所需** |
| 5 | 操作点随模型版本失效(v1 的 0.855 用在 v2/v3 上次优) | 阈值是分数分布的函数,分布随训练数据变 | 每个模型版本重新扫描;最终配置写入 ONNX metadata + 代码默认值,并由本篇审计 cell 断言一致 | **操作点是"模型版本"的属性;配置的唯一事实源头是模型自带的 metadata** |
| 6 | 审计 cell 首跑即报"阈值漂移" | metadata 写在量化之前的 fp32 上,而 quantize_static 重建图时不保留源 metadata(只写自己的工具章)| 盖章动作移到流水线末端,量化后对全部产物统一执行 | 配置注入要放在流水线最后一个会重写文件的环节之后;"我以为会继承"必须被验证,而这正是审计 cell 的工作——它上岗第一次运行就抓到了真问题,回归检查的价值即刻兑现 |

## 已知限制(诚实清单)

1. 合成流仅含 6 种背景噪声;真实声学多样性更大
2. 负例流中没有"其他人说话"的语音——对话场景 FA/hour 会高于 2.0,
   下一步应在流中混入负例单词序列重测
3. 单 seed、单条 30 min 流;严格做法是多 seed 多流取置信区间
4. FA/hour=2 仍高于商用唤醒词(通常 <0.1);但本系统 20k 参数、37.5 KB,
   处于极小模型档位,且尚有明确的改进路径(见下)

## 后续路线(按性价比排序)

1. `record_wakewords.py` 实录 "hey computer",换真唤醒词全流程重跑——
   检验小样本 + 波形增强的价值(nb02 caveat 的正式实验)
2. 数据修复第二轮:--snr-low 5 重生成带噪正例 + pos_weight 改数缓存
3. 负例语音流的 FA/hour 评测(限制 #2)
4. 增量 STFT(消除 90% 重叠帧的重复计算,10× 特征提取预算)
5. 截尾 hard negative("yes" 去掉 /s/)——若换唤醒词后 confusable 复现

---

## 项目总结:四本 notebook 对应岗位三项要求

| 岗位要求 | 本项目证据 |
|---|---|
| Data collection | 断点续传下载器、数据完整性验证、噪声负例/带噪正例的定向生成(含泄漏防护)、record_wakewords.py |
| ML feature development | from-scratch DSP 链(与 torchaudio 逐 bin 交叉验证)、增强 bug 的可视化发现与修复、特征缓存 11× 提速、三轮数据分布迭代 |
| Automated toolchain validation | 七段门禁导出/量化流水线、parity/判决一致性/配置一致性三层自动审计、五个由异常数字揪出的工具链陷阱 |

最终交付物:37.5 KB int8 ONNX(自带部署配置 metadata),流式 recall 0.900 / FA 2.0/hour @ 10 dB SNR,
单核推理 0.15 ms,67 个 pytest 全绿。